# Ellipsoid Fitting — Google Colab Demo
## Li & Griffiths (2004) Least-Squares Algorithm

This notebook provides a **Colab-ready** demonstration of the
[Li & Griffiths (2004)](https://doi.org/10.1109/GMAP.2004.1290055)
ellipsoid-specific least-squares fitting algorithm with:

- **Interactive 3-D Plotly visualisation** of the point cloud and fitted surface
- **Comprehensive fitting accuracy metrics** (algebraic RMSE, geometric approximation, etc.)
- **Fully reproducible**: seeded RNG, runs end-to-end from a fresh kernel

### Quick start
1. Click **Runtime → Run all** (or press `Ctrl+F9`).
2. All dependencies will be installed automatically in the first cell.
3. Scroll to the **Interactive 3-D Visualisation** section to explore the fit.

### Contents
1. [Setup & dependencies](#1-setup--dependencies)
2. [Generate synthetic point cloud](#2-generate-synthetic-point-cloud)
3. [Fit the ellipsoid](#3-fit-the-ellipsoid)
4. [Fitting accuracy metrics](#4-fitting-accuracy-metrics)
5. [Interactive 3-D visualisation](#5-interactive-3-d-visualisation)
6. [Bonus: rotated ellipsoid & CSV data](#6-bonus-rotated-ellipsoid--csv-data)
7. [How to interpret the results](#7-how-to-interpret-the-results)

## 1. Setup & dependencies

In [ ]:
# Install required packages (Colab may already have numpy/scipy/matplotlib)
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

_pip('plotly>=5.0', 'ipywidgets>=7.0', 'scipy>=1.8')

# Clone / mount the repository so we can import the package
import os

_REPO_URL = (
    'https://github.com/QL-UoHull/'
    'Ellipsoid-Fitting-via-Least-Squares-with-Ellipsoid-Specific-Constraints-'
    'Li-Griffiths-2004-.git'
)
_REPO_DIR = 'ellipsoid_repo'

if not os.path.isdir(_REPO_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1', _REPO_URL, _REPO_DIR])
    print(f'Repository cloned to {_REPO_DIR}/')
else:
    print(f'Repository already present at {_REPO_DIR}/')

sys.path.insert(0, _REPO_DIR)
print('Setup complete.')

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Enable Plotly rendering inside Colab / JupyterLab
import plotly.io as pio
pio.renderers.default = 'colab'

from ellipsoid_fitting import (
    fit_ellipsoid,
    generate_ellipsoid_points,
    algebraic_distance,
    residuals_rms,
)

print('All imports successful!')

## 2. Generate synthetic point cloud

We use the Fibonacci sphere algorithm to place points quasi-uniformly on an
ellipsoid surface, then add Gaussian noise. The RNG is seeded for reproducibility.

In [ ]:
# Ground-truth parameters (change these to experiment)
TRUE_CENTRE = (1.0, 2.0, 3.0)
TRUE_RADII  = (5.0, 3.0, 2.0)
N_POINTS    = 300
NOISE_STD   = 0.05
SEED        = 42

pts = generate_ellipsoid_points(
    centre=TRUE_CENTRE,
    radii=TRUE_RADII,
    n_points=N_POINTS,
    noise_std=NOISE_STD,
    seed=SEED,
)

print(f'Generated {len(pts)} points')
print(f'x ∈ [{pts[:,0].min():.2f}, {pts[:,0].max():.2f}]')
print(f'y ∈ [{pts[:,1].min():.2f}, {pts[:,1].max():.2f}]')
print(f'z ∈ [{pts[:,2].min():.2f}, {pts[:,2].max():.2f}]')

## 3. Fit the ellipsoid

In [ ]:
x, y, z = pts[:, 0], pts[:, 1], pts[:, 2]
result = fit_ellipsoid(x, y, z)

print('=' * 55)
print('  Li–Griffiths Ellipsoid Fit')
print('=' * 55)
print(f"  True  centre : {np.array(TRUE_CENTRE)}")
print(f"  Fitted centre: {result['centre'].round(4)}")
print()
print(f"  True  radii  : {np.sort(TRUE_RADII)[::-1]}")
print(f"  Fitted radii : {result['radii'].round(4)}")
print()
print(f"  RMS algebraic residual: {residuals_rms(x, y, z, result):.6f}")
print('=' * 55)

## 4. Fitting accuracy metrics

We compute several error statistics based on the algebraic residual
`F(x, y, z) = Ax² + By² + ... + J`, which equals zero for exact ellipsoid points.

In [ ]:
# Algebraic residuals
alg_errors = algebraic_distance(x, y, z, result['coefficients'])

mean_err = float(np.mean(alg_errors))
std_err  = float(np.std(alg_errors))
rmse     = float(np.sqrt(np.mean(alg_errors ** 2)))
max_abs  = float(np.max(np.abs(alg_errors)))

# Normalised residuals (geometric approximation)
A, B, C_c, D, E, F_c, G, H, I_c, J = result['coefficients']
grad_x = 2*A*x + 2*F_c*y + 2*E*z + 2*G
grad_y = 2*F_c*x + 2*B*y + 2*D*z + 2*H
grad_z = 2*E*x + 2*D*y + 2*C_c*z + 2*I_c
grad_norm = np.sqrt(grad_x**2 + grad_y**2 + grad_z**2)
norm_res = alg_errors / np.where(grad_norm > 1e-12, grad_norm, np.nan)
valid = ~np.isnan(norm_res)

print('=' * 60)
print('  FITTING ACCURACY METRICS')
print('=' * 60)
print(f'  N points             : {len(x)}')
print()
print('  --- Algebraic residuals F(x,y,z) ---')
print(f'  Mean                 : {mean_err:+.6f}')
print(f'  Std                  :  {std_err:.6f}')
print(f'  RMSE                 :  {rmse:.6f}')
print(f'  Max |error|          :  {max_abs:.6f}')
print()
print('  --- Normalised residuals (geom. approx.) ---')
print(f'  Mean                 : {np.mean(norm_res[valid]):+.6f}')
print(f'  Std                  :  {np.std(norm_res[valid]):.6f}')
print(f'  RMSE                 :  {np.sqrt(np.mean(norm_res[valid]**2)):.6f}')
print()
print('  --- Geometric parameters ---')
print(f"  Fitted centre        : {result['centre'].round(4)}")
print(f"  Fitted radii         : {result['radii'].round(4)}")
print(f'  True centre          : {np.array(TRUE_CENTRE)}')
print(f'  True radii (desc.)   : {np.sort(TRUE_RADII)[::-1]}')
print(f'  Centre error (L2)    : {np.linalg.norm(result["centre"] - np.array(TRUE_CENTRE)):.6f}')
print(f'  Radii error (L2)     : {np.linalg.norm(np.sort(result["radii"])[::-1] - np.sort(TRUE_RADII)[::-1]):.6f}')
print('=' * 60)

In [ ]:
# Residual histogram
fig_hist = go.Figure()
fig_hist.add_trace(go.Histogram(
    x=alg_errors, nbinsx=40,
    marker_color='steelblue', opacity=0.75,
    name='Algebraic residuals',
))
fig_hist.add_vline(
    x=mean_err, line_dash='dash', line_color='red',
    annotation_text=f'Mean={mean_err:.4f}',
    annotation_position='top right',
)
fig_hist.update_layout(
    title='Distribution of Algebraic Residuals F(x,y,z)',
    xaxis_title='Algebraic residual',
    yaxis_title='Count',
    bargap=0.05,
    height=380,
)
fig_hist.show()

## 5. Interactive 3-D visualisation

The plot below renders directly in Colab. Use your mouse to:
- **Rotate**: click and drag
- **Zoom**: scroll wheel
- **Pan**: right-click drag (or Shift + drag)
- **Toggle layers**: click legend items

In [ ]:
def make_ellipsoid_mesh(centre, radii, axes_mat, n_u=60, n_v=30):
    """Return (X, Y, Z) grid arrays for a parameterised ellipsoid surface."""
    u = np.linspace(0, 2 * np.pi, n_u)
    v = np.linspace(0, np.pi, n_v)
    xs = radii[0] * np.outer(np.cos(u), np.sin(v))
    ys = radii[1] * np.outer(np.sin(u), np.sin(v))
    zs = radii[2] * np.outer(np.ones_like(u), np.cos(v))
    shape = xs.shape
    pts_m = np.column_stack([xs.ravel(), ys.ravel(), zs.ravel()]) @ axes_mat.T
    cx, cy, cz = centre
    X = pts_m[:, 0].reshape(shape) + cx
    Y = pts_m[:, 1].reshape(shape) + cy
    Z = pts_m[:, 2].reshape(shape) + cz
    return X, Y, Z


X_ell, Y_ell, Z_ell = make_ellipsoid_mesh(
    result['centre'], result['radii'], result['axes']
)

fig_3d = go.Figure()

fig_3d.add_trace(go.Scatter3d(
    x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
    mode='markers',
    marker=dict(size=2.5, color='steelblue', opacity=0.6),
    name='Noisy point cloud',
))

fig_3d.add_trace(go.Surface(
    x=X_ell, y=Y_ell, z=Z_ell,
    colorscale='Oranges',
    opacity=0.35,
    showscale=False,
    name='Fitted ellipsoid',
))

fig_3d.update_layout(
    title=(
        f'Li–Griffiths Ellipsoid Fit — '
        f'RMSE={rmse:.5f}, '
        f'CentreΔ={np.linalg.norm(result["centre"] - np.array(TRUE_CENTRE)):.4f}, '
        f'RadiiΔ={np.linalg.norm(np.sort(result["radii"])[::-1] - np.sort(TRUE_RADII)[::-1]):.4f}'
    ),
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='z',
        aspectmode='data',
    ),
    legend=dict(x=0.02, y=0.98),
    margin=dict(l=0, r=0, b=0, t=60),
    height=600,
)
fig_3d.show()

In [ ]:
# Side-by-side: residual colour-coded point cloud
# Points are coloured by their algebraic residual magnitude
abs_err = np.abs(alg_errors)

fig_compare = go.Figure()

fig_compare.add_trace(go.Scatter3d(
    x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
    mode='markers',
    marker=dict(
        size=3,
        color=abs_err,
        colorscale='Viridis',
        colorbar=dict(title='|F(x,y,z)|'),
        opacity=0.85,
    ),
    name='Residual-coloured points',
))

fig_compare.add_trace(go.Surface(
    x=X_ell, y=Y_ell, z=Z_ell,
    colorscale='Oranges',
    opacity=0.25,
    showscale=False,
    name='Fitted ellipsoid',
))

fig_compare.update_layout(
    title='Points coloured by algebraic residual magnitude |F(x,y,z)|',
    scene=dict(
        xaxis_title='x', yaxis_title='y', zaxis_title='z',
        aspectmode='data',
    ),
    margin=dict(l=0, r=0, b=0, t=60),
    height=600,
)
fig_compare.show()

## 6. Bonus: rotated ellipsoid & CSV data

The algorithm handles arbitrary orientations without any modification.

In [ ]:
from scipy.spatial.transform import Rotation

R = Rotation.from_euler('xyz', [30, 45, 60], degrees=True).as_matrix()
pts_rot = generate_ellipsoid_points(
    centre=(0.0, 0.0, 0.0),
    radii=(6.0, 4.0, 2.5),
    rotation=R,
    n_points=400,
    noise_std=0.10,
    seed=7,
)

result_rot = fit_ellipsoid(pts_rot[:, 0], pts_rot[:, 1], pts_rot[:, 2])
print(f"True  radii : {np.sort([6.0, 4.0, 2.5])[::-1]}")
print(f"Fitted radii: {result_rot['radii'].round(3)}")
print(f"Centre      : {result_rot['centre'].round(3)}")
print(f"RMS residual: {residuals_rms(pts_rot[:,0], pts_rot[:,1], pts_rot[:,2], result_rot):.4f}")

X_rot, Y_rot, Z_rot = make_ellipsoid_mesh(
    result_rot['centre'], result_rot['radii'], result_rot['axes']
)

fig_rot = go.Figure()
fig_rot.add_trace(go.Scatter3d(
    x=pts_rot[:, 0], y=pts_rot[:, 1], z=pts_rot[:, 2],
    mode='markers',
    marker=dict(size=2.5, color='steelblue', opacity=0.6),
    name='Point cloud (rotated)',
))
fig_rot.add_trace(go.Surface(
    x=X_rot, y=Y_rot, z=Z_rot,
    colorscale='Greens',
    opacity=0.30,
    showscale=False,
    name='Fitted ellipsoid',
))
fig_rot.update_layout(
    title='Rotated ellipsoid fit',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z',
               aspectmode='data'),
    height=550,
    margin=dict(l=0, r=0, b=0, t=40),
)
fig_rot.show()

In [ ]:
# CSV datasets bundled with the repository
data_dir = os.path.join(_REPO_DIR, 'data')

for fname in sorted(os.listdir(data_dir)):
    if fname.endswith('.csv'):
        path = os.path.join(data_dir, fname)
        data = np.loadtxt(path, delimiter=',', skiprows=1)
        r = fit_ellipsoid(data[:, 0], data[:, 1], data[:, 2])
        rms = residuals_rms(data[:, 0], data[:, 1], data[:, 2], r)
        print(f'{fname}')
        print(f"  centre : {r['centre'].round(3)}")
        print(f"  radii  : {r['radii'].round(3)}")
        print(f'  RMS    : {rms:.5f}')
        print()

## 7. How to interpret the results

### The 3-D Plot

| Element | Description |
|---------|-------------|
| **Blue points** | Raw 3-D input point cloud. Noise causes scatter around the true surface. |
| **Orange/Green surface** | The fitted ellipsoid surface. Ideally it passes through all blue points. |
| **Coloured points** | In the residual plot, colour encodes the algebraic error magnitude at each point — dark purple = near zero (good fit), yellow = larger error. |

### The Accuracy Metrics

| Metric | What it means |
|--------|---------------|
| **Algebraic RMSE** | Root-mean-square of `F(x,y,z)` at each input point. Values ≪ 1 (relative to data scale) indicate a good fit. |
| **Mean algebraic error** | Should be near 0 for unbiased noise. A large absolute mean suggests a systematic offset. |
| **Std of errors** | Spread of algebraic residuals; grows with noise level. |
| **Normalised residuals** | Algebraic error ÷ gradient magnitude — a geometric distance approximation. |
| **Centre Δ (L2)** | Euclidean distance between fitted centre and the true centre. |
| **Radii Δ (L2)** | Norm of the difference between fitted and true semi-axis lengths. |

### Rules of thumb
- Algebraic RMSE < 0.001 (for data with unit-scale radii) indicates an excellent fit.
- Centre and radii errors are more interpretable when true parameters are known.
- Increasing `NOISE_STD` in Section 2 and re-running will degrade all metrics predictably.